In [5]:
import json
import matplotlib.pyplot as plt
import pandas as pd


In [9]:
entsoe_timeseries_df = pd.read_csv("../datasets/ENTSOE_power_time_series.csv", index_col=0, parse_dates=True)
with open("../datasets/metadata_wind_farms.json") as f:
    wind_farms_metadata = json.load(f)["wind_farms"]

In [16]:
df_KNMI_stations = {}
for wind_farm in wind_farms_metadata:
    station = wind_farm.get("nearest_KNMI_station")
    try:
        df = pd.read_csv
        df_KNMI_stations[station] = df
    except:
        print(f"No file found for station {wind_farm['id']}")

No file found for station Borssele_12
No file found for station Borssele_34
No file found for station Gemini
No file found for station Hollandse_Kust_Zuid
No file found for station Hollandse_Kust_Noord
No file found for station Windfarm_Princess_Amalia
No file found for station OWEZ


In [ ]:
#fitting power curves
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def logistic_piece(v, alpha, beta, gamma, delta):
    return alpha / (1 + np.exp(-gamma * (v - delta))) ** (1 / beta)

def piecewise_logistic(v, alpha1, beta1, gamma1, delta1, alpha2, beta2, gamma2, delta2, tau):
    return np.where(
        v < tau,
        logistic_piece(v, alpha1, beta1, gamma1, delta1),
        logistic_piece(v, alpha2, beta2, gamma2, delta2)
    )

# --- fit ---
# drop rows where turbine is off or wind speed is 0
fit_df = df[(df["Turn_off"] == 0) & (df["Scaled_Windspeed"] > 0)].copy()

v = fit_df["Scaled_Windspeed"].values
p = fit_df["Power"].values

# initial guesses based on the paper's values
p0 = [
    0.001,   # alpha1
    0.023,   # beta1
    -0.128,  # gamma1
    0.143,   # delta1
    0.005,   # alpha2
    -0.264,  # beta2
    4.244,   # gamma2
    -19.066, # delta2
    9.0      # tau
]

popt, pcov = curve_fit(
    piecewise_logistic,
    v, p,
    p0=p0,
    maxfev=10000
)

print("Fitted parameters:")
print(f"alpha1={popt[0]:.3f}, beta1={popt[1]:.3f}, gamma1={popt[2]:.3f}, delta1={popt[3]:.3f}")
print(f"alpha2={popt[4]:.3f}, beta2={popt[5]:.3f}, gamma2={popt[6]:.3f}, delta2={popt[7]:.3f}")
print(f"tau={popt[8]:.3f}")

# --- apply cut-out at 25 m/s ---
v_range = np.linspace(0, 30, 300)
p_fitted = np.where(
    v_range <= 25,
    piecewise_logistic(v_range, *popt),
    0
)

# --- plot ---
plt.figure(figsize=(8, 5))
plt.scatter(v, p, s=1, alpha=0.2, label="Observed")
plt.plot(v_range, p_fitted, color="red", linewidth=2, label="Fitted curve")
plt.axvline(25, color="gray", linestyle="--", label="Cut-out (25 m/s)")
plt.xlabel("Wind speed (m/s)")
plt.ylabel("Power (MW)")
plt.title("Fitted piecewise logistic power curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def predict_power(wind_speed, popt, cut_out=25.0):
    p = piecewise_logistic(wind_speed, *popt)
    p = np.where(wind_speed >= cut_out, 0, p)  # cut-out
    p = np.where(wind_speed < 3, 0, p)          # cut-in (optional)
    p = np.maximum(p, 0)                         # no negative power
    return p

# single value
predict_power(8.5, popt)

# array / dataframe column
df["Power_predicted"] = predict_power(df["Scaled_Windspeed"].values, popt)